# 06 — Remaining Useful Life (RUL) Prediction

Trains and compares Random Forest, Gradient Boosting, and XGBoost regressors to estimate the Remaining Useful Life (in days) of each piece of equipment.

This notebook wraps the logic in `src/train_rul_model.py` — run that script directly for a clean, reproducible pipeline run.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))

import train_rul_model

train_rul_model.main()

2026-09-25 11:02:05,774 | INFO | Loading + cleaning + feature-engineering data...


2026-09-25 11:02:05,836 | INFO | Starting data cleaning: 22440 rows, 26 columns


2026-09-25 11:02:05,871 | INFO |   Removed 440 exact duplicate rows


2026-09-25 11:02:05,872 | INFO |   Temperature_C: flagging 132 out-of-range readings as missing


2026-09-25 11:02:05,875 | INFO |   Pressure_Bar: flagging 197 out-of-range readings as missing


...

2026-09-25 11:03:52,612 | INFO |   Gradient Boosting Regressor  MAE=37.19 RMSE=45.43 R2=0.4625


2026-09-25 11:03:54,336 | INFO |   XGBoost Regressor            MAE=37.65 RMSE=45.99 R2=0.4493


2026-09-25 11:03:54,341 | INFO | Selected final RUL model: Gradient Boosting Regressor


2026-09-25 11:03:54,348 | INFO | Saved RUL model -> /home/claude/oil_gas_predictive_maintenance/models/rul_prediction_model.pkl



                      Model   MAE  RMSE     R2
Gradient Boosting Regressor 37.19 45.43 0.4625
          XGBoost Regressor 37.65 45.99 0.4493
    Random Forest Regressor 37.78 46.20 0.4442



## Review RUL model comparison report

In [2]:
import pandas as pd
pd.read_csv('../reports/rul_model_performance.csv')

,Model,MAE,RMSE,R2
0,Gradient Boosting Regressor,37.19,45.43,0.4625
1,XGBoost Regressor,37.65,45.99,0.4493
2,Random Forest Regressor,37.78,46.20,0.4442


## Predicted vs actual RUL (test-set style spot check)

In [3]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))

import joblib
import plotly.express as px
import plotly.io as pio
pio.templates.default = 'plotly_dark'

from data_preprocessing import clean_raw_dataframe
from feature_engineering import add_engineered_features, ALL_MODEL_FEATURES
from train_failure_model import encode_features

raw = pd.read_csv('../data/raw/oil_gas_equipment_sensor_data.csv').sample(500, random_state=42)
clean = clean_raw_dataframe(raw, verbose=False)
fe = add_engineered_features(clean)

encoders = joblib.load('../models/encoder.pkl')
X, _ = encode_features(fe[ALL_MODEL_FEATURES], encoders=encoders)

rul_model = joblib.load('../models/rul_prediction_model.pkl')
preds = rul_model.predict(X)

compare = pd.DataFrame({'Actual_RUL': fe['Remaining_Useful_Life'].values, 'Predicted_RUL': preds})
px.scatter(compare, x='Actual_RUL', y='Predicted_RUL', title='Predicted vs Actual Remaining Useful Life')

2026-09-25 11:03:54,598 | INFO |   Removed 0 exact duplicate rows


2026-09-25 11:03:54,599 | INFO |   Temperature_C: flagging 6 out-of-range readings as missing


2026-09-25 11:03:54,601 | INFO |   Pressure_Bar: flagging 4 out-of-range readings as missing


2026-09-25 11:03:54,603 | INFO |   Vibration_mm_s: flagging 6 out-of-range readings as missing


2026-09-25 11:03:54,604 | INFO |   RPM: flagging 6 out-of-range readings as missing


2026-09-25 11:03:54,606 | INFO |   Motor_Current_A: flagging 4 out-of-range readings as missing


2026-09-25 11:03:54,608 | INFO |   Oil_Pressure_Bar: flagging 2 out-of-range readings as missing


2026-09-25 11:03:54,610 | INFO |   Power_Consumption_kW: flagging 10 out-of-range readings as missing
